# ¿Falló el término, o le faltó coeficiente? — la medición

Este cuaderno **corre** el diagnóstico y nada más: no arma ninguna tabla ni conclusión — eso vive en `Benchmark_Noise_Diagnostic_Report_v1.ipynb`, que lee el `diagnostic.json` que esto deja y lo presenta.

Los techos de la campaña se buscaron con el material **limpio** y se mantienen fijos en los cinco niveles de contaminación. Esa decisión es deliberada y tiene un costo declarado: **una caída no se puede atribuir.** El término fallando bajo ruido y el coeficiente quedándose corto producen exactamente la misma curva. Esto es lo barato que separa las dos.

> **Tres puntos, y se paga uno solo.**
>
> Los dos brazos a este nivel bajo el techo limpio ya salen del barrido — están medidos y no se vuelven a correr. Lo que cuesta es la búsqueda del techo *sobre material contaminado*. Si el techo re-buscado recupera lo perdido, fue el coeficiente; si no lo recupera, fue el término.
>
> **`D` y `G`, y nadie más.** Son los dos métodos completos, uno por familia, y los únicos que llevan el coeficiente. `A` y `B` no tienen término de adaptación al que re-buscarle un techo, y `C`, `E` y `F` son ablaciones que multiplicarían la búsqueda sin agregar diagnóstico.
>
> **Sobre la transferencia de la curva y ninguna otra.** Buscar sobre las seis costaría seis veces lo que el diagnóstico vale y mediría cinco transferencias que la curva nunca recorrió, así que no habría contra qué leerlas.
>
> **El nivel es el tope del rango, fijado antes de que la curva existiera.** En el extremo el coeficiente está bajo la máxima presión, así que un techo re-buscado que no recupera nada ahí no recupera nada en ningún lado — y la lectura no depende de dónde alguien eligió mirar.
>
> **Sus números son de diagnóstico y no entran en las tablas del veredicto:** lo único que deciden es si vale reestructurar para techos por nivel. La re-búsqueda que se paga acá no gobierna ningún registro — `governs_the_ceilings_record` es falso bajo contaminación y sobre una transferencia sola.

In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json

from MIL_CREDA_Benchmark import config, contamination, harness

tasa = config.NOISE_DIAGNOSTIC_LEVEL
device = harness.resolve_device()
# La escala configurada decide si esto es un ensayo, y el ensayo decide DÓNDE
# escribe --- la misma lectura y la misma razón que en la campaña y en el
# barrido. Escrito bajo `Results/Noise/` a secas, un diagnóstico de ensayo
# pisaría el de la corrida completa con números que no se pueden citar; el paso
# que corre este cuaderno se niega cuando la escala es la completa, porque sus
# `produces` nombran el árbol de ENSAYO y ninguno más.
ES_ENSAYO = config.is_pilot_scale()
# Y de qué árbol se LEE lo que dejó el barrido, que es otra pregunta. Las dos son
# la misma en el recorrido local y se separan en el ENSAYO REMOTO: ahí el
# diagnóstico corre a escala reducida, en el worker, contra la línea limpia que
# el barrido dejó a escala COMPLETA. Fuera de ese modo esto es `is_pilot_scale()`
# y no cambia nada.
ES_ENSAYO_ENTRADA = config.upstream_pilot_scale()
reduccion = harness.Reduction(device=str(device), environment=harness.environment(),
                              labelNoise=tasa, pilot=ES_ENSAYO)

print(f"nivel de diagnóstico: ρ={tasa:g} "
      f"(el tope de {[f'{r:g}' for r in config.NOISE_LEVELS]})")
print("transferencia: {}->{}".format(*config.NOISE_TRANSFER))
print(f"brazos: {[config.NAME_OF[a] for a in config.NOISE_DIAGNOSTIC_ARMS]}")
print("escala:", "ensayo" if ES_ENSAYO else "completa",
      "| escribe en", config.noise_axis_for(reduccion.pilot))

## La medición

El techo buscado **sobre material contaminado**: es el punto que la campaña no
tiene y la única razón por la que este cuaderno cuesta algo. Una sola búsqueda, y
ninguna campaña — los otros dos puntos ya están en el registro del barrido.

In [ ]:
buscado = harness.search_ceilings(reduccion, device, noise=tasa,
                                  transfers=[config.NOISE_TRANSFER],
                                  pilot=reduccion.pilot)

## El registro

Lo que este cuaderno midió, escrito donde su paso declara que vive. El otro
extremo se lee y no se vuelve a correr: si esta celda y el barrido dijeran cosas
distintas habría dos versiones del mismo número.

In [ ]:
registro = {
    "level": tasa,
    "transfer": "{}->{}".format(*config.NOISE_TRANSFER),
    "arms": list(config.NOISE_DIAGNOSTIC_ARMS),
    "searchedUnderNoise": buscado,
    # Del BARRIDO y no de una campaña: a este nivel no hay campaña completa ---
    # sólo 0.0 y NOISE_REPORTED la tienen --- y la comparación es sobre la
    # transferencia del barrido de todos modos.
    #
    # Y del barrido de la escala de ENTRADA. Sin coordenada ninguna leía el que
    # rige, así que un diagnóstico de ensayo ---escrito bajo `Pilot/`, con su
    # re-búsqueda medida a tres épocas--- guardaba adentro el resumen del barrido
    # COMPLETO como su línea limpia, sin decirlo. Toda la pregunta de este
    # cuaderno es la diferencia entre esos dos números, así que mezclar escalas
    # POR ACCIDENTE es comparar el término, el coeficiente y la escala a la vez,
    # y las tres se ven igual de plausibles en la tabla que sale.
    #
    # La coordenada decía `reduccion.pilot`. En el recorrido local eso sigue
    # siendo lo mismo que dice esta lectura, y por eso el defecto sigue cerrado.
    # Lo que cambia es el ENSAYO REMOTO, donde esa mezcla deja de ser un
    # accidente y pasa a ser lo que se está probando: la re-búsqueda reducida
    # contra la línea limpia que el barrido dejó a escala completa, que es la que
    # el diagnóstico real va a leer. Y no queda sin decir: `pilot` viaja en la
    # reducción sellada al lado, así que la tabla sabe a qué escala se midió cada
    # mitad.
    "cleanCeilingRun": (contamination.load(tasa, kind="curve",
                                           pilot=ES_ENSAYO_ENTRADA)
                        or {}).get("summary"),
    "revision": config.REVISION,
    "diagnosticOnly": ("estos números no entran en las tablas del veredicto; "
                       "deciden si vale reestructurar para techos por nivel"),
}
# Bajo la raíz de ESTA corrida, compuesta desde la reducción con la que se
# midió y no desde una constante.
destino = config.noise_axis_for(reduccion.pilot)
destino.mkdir(parents=True, exist_ok=True)
(destino / "diagnostic.json").write_text(
    json.dumps(registro, indent=2, default=str), encoding="utf-8")

print("escrito:", destino / "diagnostic.json")
print()
print("la tabla y la conclusión las arma "
      "Benchmark_Noise_Diagnostic_Report_v1.ipynb")